# 01 · Comprensión y auditoría de datos

Esta auditoría respalda la tarjeta #2. Se leen los tres archivos por bloques para revisar el esquema, la completitud y las coincidencias de filas sin cargar los CSV completos a la vez. La exploración del puntaje se limita a entrenamiento. Se requiere Python con pandas y NumPy.

In [1]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd

carpeta = next(p for p in (Path("Datasets"), Path("../Datasets")) if p.is_dir())
archivos = {nombre: carpeta / f"{nombre}.csv" for nombre in ("train", "validation", "test")}

def leer_columnas(ruta):
    with ruta.open(encoding="utf-8-sig", newline="") as archivo:
        return next(csv.reader(archivo))

columnas = leer_columnas(archivos["train"])
for nombre, ruta in archivos.items():
    assert leer_columnas(ruta) == columnas, f"Esquema distinto en {nombre}"

print(f"Los tres archivos comparten {len(columnas)} columnas en el mismo orden.")
print(", ".join(columnas))

Los tres archivos comparten 21 columnas en el mismo orden.
Age, Grade, Gender, Race, SES_Quartile, ParentalEducation, SchoolType, Locale, TestScore_Math, TestScore_Reading, TestScore_Science, GPA, AttendanceRate, StudyHours, InternetAccess, Extracurricular, PartTimeJob, ParentSupport, Romantic, FreeTime, GoOut


## Completitud y coincidencias

Cada huella de 64 bits representa una fila completa. Las coincidencias son posibles duplicados o cruces de registros y pueden confirmarse comparando las filas originales. El método no identifica si dos filas diferentes corresponden al mismo estudiante, ya que el archivo carece de un identificador.

In [2]:
resumen = []
huellas_unicas = {}
puntuaciones = []

for nombre, ruta in archivos.items():
    faltantes = pd.Series(0, index=columnas, dtype="int64")
    bloques_huellas = []
    for bloque in pd.read_csv(ruta, dtype=str, chunksize=150_000, encoding="utf-8-sig"):
        faltantes += bloque.isna().sum()
        bloques_huellas.append(pd.util.hash_pandas_object(bloque, index=False).to_numpy())
        if nombre == "train":
            puntuaciones.append(pd.to_numeric(bloque["TestScore_Math"], errors="coerce").to_numpy())

    huellas = np.concatenate(bloques_huellas)
    unicas = np.unique(huellas)
    huellas_unicas[nombre] = unicas
    detalle = faltantes[faltantes > 0].to_dict()
    resumen.append({
        "archivo": nombre,
        "filas": len(huellas),
        "MiB": round(ruta.stat().st_size / 1024**2, 1),
        "celdas_faltantes": int(faltantes.sum()),
        "faltantes_por_columna": detalle,
        "filas_repetidas_adicionales": len(huellas) - len(unicas),
    })

print(pd.DataFrame(resumen).to_string(index=False))

from itertools import combinations
for primero, segundo in combinations(huellas_unicas, 2):
    coincidencias = np.intersect1d(
        huellas_unicas[primero], huellas_unicas[segundo], assume_unique=True
    )
    print(f"{primero} / {segundo}: {len(coincidencias):,} huellas únicas compartidas")

   archivo   filas    MiB  celdas_faltantes faltantes_por_columna  filas_repetidas_adicionales
     train 8000774 1267.1                 0                    {}                            0
validation  999229  158.2                 0                    {}                            0
      test  999997  158.4                 0                    {}                            0
train / validation: 0 huellas únicas compartidas
train / test: 0 huellas únicas compartidas
validation / test: 0 huellas únicas compartidas


In [3]:
puntaje = pd.Series(np.concatenate(puntuaciones), name="TestScore_Math")
print(puntaje.describe(percentiles=[0.10, 0.20, 0.25, 0.50, 0.75, 0.90]).round(2))
print(f"Sin puntaje numérico: {puntaje.isna().sum():,}")
print(f"Fuera del rango exploratorio 0–100: {(puntaje.notna() & ~puntaje.between(0, 100)).sum():,}")

count    8000774.00
mean          74.98
std            9.94
min           22.58
10%           62.18
20%           66.57
25%           68.25
50%           75.00
75%           81.74
90%           87.82
max          100.00
Name: TestScore_Math, dtype: float64
Sin puntaje numérico: 0
Fuera del rango exploratorio 0–100: 0


## Decisiones pendientes

El puntaje de Matemáticas se utilizará para construir la etiqueta. El punto de corte se justificará con documentación de la fuente o con una regla exploratoria explícita basada en entrenamiento. Para usar GPA, asistencia u otros puntajes como predictores, primero se debe verificar su disponibilidad antes de la alerta. Todavía queda por documentar la procedencia, los rangos de las demás columnas y la representatividad de los datos.

## Diccionario y criterios de uso

| Variables | Significado | Uso propuesto |
|---|---|---|
| Age, Grade | Edad y nivel escolar | Grade como candidato. Age se reserva para auditoría. |
| Gender, Race, SES_Quartile | Género, categoría racial y cuartil socioeconómico | Auditoría agregada de equidad. |
| ParentalEducation, SchoolType, Locale | Educación parental, tipo de escuela y contexto territorial | Descripción y auditoría de representatividad. |
| TestScore_Math | Puntaje de Matemáticas | Construcción de la etiqueta. |
| TestScore_Reading, TestScore_Science, GPA | Puntajes de Lectura y Ciencias y promedio académico | Excluir del primer modelo por su temporalidad incierta y la posible inclusión de Matemáticas en GPA. |
| AttendanceRate | Proporción de asistencia | Reservar hasta confirmar que corresponde a un período anterior al puntaje. |
| StudyHours, Extracurricular | Horas de estudio (período sin documentar) y participación extracurricular | Candidatos si se recogen antes de la evaluación. |
| InternetAccess, PartTimeJob, ParentSupport | Acceso a internet, trabajo parcial y apoyo parental | Descripción de contexto. |
| Romantic, FreeTime, GoOut | Relación sentimental, tiempo libre y frecuencia de salidas según el nombre de las variables | Excluir del modelo inicial. Las escalas carecen de documentación. |

Los significados se infieren de los nombres y valores de las columnas. Las reglas siguientes son controles exploratorios. Los límites observados de GPA, edad y curso no certifican una escala oficial.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

from collections import Counter

numericas = ["Age", "Grade", "SES_Quartile", "TestScore_Math", "TestScore_Reading",
             "TestScore_Science", "GPA", "AttendanceRate", "StudyHours",
             "FreeTime", "GoOut"]
binarias = ["InternetAccess", "Extracurricular", "PartTimeJob", "ParentSupport", "Romantic"]
categoricas = ["Gender", "Race", "ParentalEducation", "SchoolType", "Locale"]
reglas = {c: (0, 100) for c in numericas if c.startswith("TestScore_")}
reglas.update({"AttendanceRate": (0, 1), "StudyHours": (0, float("inf")),
               "SES_Quartile": (1, 4)})
reglas.update({c: (0, 1) for c in binarias})
enteras = ["Age", "Grade", "SES_Quartile", "FreeTime", "GoOut"] + binarias

for nombre in ("train", "validation", "test"):
    ruta = carpeta / f"{nombre}.csv"
    limites = {c: [float("inf"), float("-inf"), 0] for c in numericas + binarias}
    frecuencias = {c: Counter() for c in categoricas}
    for bloque in pd.read_csv(ruta, chunksize=250_000):
        for c, datos in limites.items():
            valores = pd.to_numeric(bloque[c], errors="coerce")
            validos = np.isfinite(valores)
            if c in reglas:
                validos &= valores.between(*reglas[c])
            if c in enteras:
                validos &= valores.eq(valores.round())
            datos[0] = min(datos[0], valores.min())
            datos[1] = max(datos[1], valores.max())
            datos[2] += int((~validos).sum())
        for c in categoricas:
            frecuencias[c].update(bloque[c].value_counts().to_dict())
    print(nombre, pd.DataFrame(limites, index=["mínimo", "máximo", "incumplimientos"]).T.to_string())
    for c, conteo in frecuencias.items():
        total = sum(conteo.values())
        print(c, {k: round(v / total * 100, 2) for k, v in conteo.most_common()})
    print()


train                       mínimo      máximo  incumplimientos
Age                14.000000   18.000000              0.0
Grade               9.000000   12.000000              0.0
SES_Quartile        1.000000    4.000000              0.0
TestScore_Math     22.576813  100.000000              0.0
TestScore_Reading  21.187972  100.000000              0.0
TestScore_Science  19.188829  100.000000              0.0
GPA                 0.713181    4.000000              0.0
AttendanceRate      0.700000    1.000000              0.0
StudyHours          0.000000    2.793829              0.0
FreeTime            1.000000    5.000000              0.0
GoOut               1.000000    5.000000              0.0
InternetAccess      0.000000    1.000000              0.0
Extracurricular     0.000000    1.000000              0.0
PartTimeJob         0.000000    1.000000              0.0
ParentSupport       0.000000    1.000000              0.0
Romantic            0.000000    1.000000              0.0
Gender {

## Procedencia y propuesta para el primer modelo

El material del proyecto carece de ficha de origen, fechas de medición, identificador de estudiante y población de referencia para estos CSV. Las categorías raciales y de educación parental sugieren un contexto distinto al del colegio objetivo. La comparación entre particiones permite describir los archivos, pero no establecer representatividad para Chile ni comprobar que el modelo anticipa rendimiento futuro.

Como definición operativa del prototipo se propone marcar como clase positiva los puntajes estrictamente inferiores al percentil 20 de entrenamiento (66,57 mostrado con dos decimales). Se conservará el valor sin redondear al aplicarlo a los tres conjuntos. El corte identifica bajo rendimiento relativo y no equivale a una nota de reprobación. Para el primer modelo se proponen Grade, StudyHours y Extracurricular. Su disponibilidad anticipada es un supuesto para la demo. Los atributos sensibles se destinan a auditoría agregada. Validación se usará para comparar modelos y elegir el umbral de alerta. Prueba queda reservada para la evaluación final.